# KAT-VAD v5 — **check C**: change maps vs the centre prior (CPU, ~15–30 min)

Rule: `core/docs/v5/PREREG_V5.md` **§3.7** (written 2026-10-30, after the v5.1 PILOT-FAIL; post-hoc, disclosed).
Training-free: reads Stage 0's stored DoTA-CAP-dev change maps (`maps/*.npz`) and scores, per clip, the S2-8 pointing
read for each map **and** for the centre prior on the **same grid and steps**.

| step | what |
|---|---|
| 1–2 | mount, env, locate Stage 0 outputs (`maps/` 569, `extract_*.json`, `REPORTS/stage0_readout.json`) |
| 3 | copy the maps to the VM (small), DoTA annotations + metadata |
| 4 | `core.tools.v5_centre_check` → `Thesis-V5/outputs/v5/v5_check_c/REPORTS/check_c_readout.{md,json}` |

**Decision (C-3):** `C14` hit − centre 95 % low > 0 → **CLIP-BEATS-CENTRE** (R7 eligible for an amendment, on the
user's word); otherwise **CENTRE-WINS** (v5 closed). G-C0: `C14` hit − chance must reproduce Stage 0 (|Δ| ≤ 1e-4).

**Drive:** code `Thesis-V5/kat-vad` (re-upload branch `v5-probe` first — needs `core/tools/v5_centre_check.py` and the
public `v5_stage0.add_read_args`); writes only to `Thesis-V5/outputs/v5/v5_check_c/`. `Thesis/`, `Thesis-V2/` read-only.
No GPU needed.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'
os.environ['PROJECT_ROOT']       = DRIVE_V5
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V5}/outputs'
os.environ['USE_TF'] = '0'

REPO = Path(f'{DRIVE_V5}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5-probe code there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_CHECK_C_DIRNAME'), 'this upload predates check C -- re-upload the code'
assert (REPO / 'core' / 'tools' / 'v5_centre_check.py').is_file(), 're-upload: v5_centre_check.py missing'

# Stage 0 ran before the Thesis-V5 layout: its outputs live under Thesis-V2 (checked first: Thesis-V5)
S0 = None
for root in (DRIVE_V5, DRIVE_V2):
    cand = Path(root) / 'outputs' / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME
    if (cand / 'maps').is_dir():
        S0 = cand
        break
assert S0 is not None, 'no Stage 0 maps/ under Thesis-V5 or Thesis-V2 outputs/v5/v5_stage0'
REPORTS_IN = sorted(S0.glob('extract_*.json'))
S0_READOUT = S0 / 'REPORTS' / 'stage0_readout.json'
n_maps = len(list((S0 / 'maps').glob('*.npz')))
print('Stage 0 dir  ', S0)
print('maps         ', n_maps, '(expect 569)')
print('reports      ', [p.name for p in REPORTS_IN])
assert n_maps == 569, n_maps
assert len(REPORTS_IN) == 5, REPORTS_IN
assert S0_READOUT.is_file(), f'missing {S0_READOUT}'

OUT = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_CHECK_C_DIRNAME / 'REPORTS'
assert OUT.is_relative_to(Path(DRIVE_V5)), OUT
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/v5cc')
LOCAL.mkdir(parents=True, exist_ok=True)
print('out          ', OUT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import av, einops, faiss; print('deps OK')"

## 3. Inputs on the VM

The 569 maps are copied once (re-runs skip it). DoTA annotations come from the DoTA GitHub zip, as in Stage 0b.

In [ ]:
import json
import shutil
import subprocess

MAPS = LOCAL / 'maps'
if len(list(MAPS.glob('*.npz'))) != n_maps:
    shutil.copytree(S0 / 'maps', MAPS, dirs_exist_ok=True)
print('maps on VM:', len(list(MAPS.glob('*.npz'))))

ANN = LOCAL / 'dota'
ANN.mkdir(exist_ok=True)
if not (ANN / 'annotations').is_dir():
    subprocess.run('curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/'
                   'DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip', shell=True, check=True, cwd=ANN)
print('annotations:', len(list((ANN / 'annotations').glob('*.json'))))

META = constants.DATA_ROOT / 'DoTA' / 'metadata_val.json'
if not META.exists():
    META = ANN / 'metadata_val.json'
    subprocess.run(['curl', '-sSL', '-o', str(META),
                    'https://raw.githubusercontent.com/MoonBlvd/Detection-of-Traffic-Anomaly/master/dataset/metadata_val.json'],
                   check=True)
SPLIT = ANN / 'val_ids.txt'
SPLIT.write_text(''.join(f'{v}\n' for v in json.loads(META.read_text())))
print('metadata:', META)

## 4. Check C (§3.7) — CPU

Expected before the run (C-7): **CENTRE-WINS**. The decision line is mechanical; paste the md back to Claude.

In [ ]:
def stream(cmd, name):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'{name} exited {proc.returncode}')


stream([sys.executable, '-m', 'core.tools.v5_centre_check',
        '--reports', *REPORTS_IN, '--maps-dir', MAPS,
        '--annotations-dir', ANN / 'annotations', '--metadata', META, '--split-file', SPLIT,
        '--stage0-readout', S0_READOUT, '--out-dir', OUT], 'v5_centre_check')
print((OUT / 'check_c_readout.md').read_text())

## 5. Record

Paste `Thesis-V5/outputs/v5/v5_check_c/REPORTS/check_c_readout.md` back to Claude, and bring
`outputs/v5/v5_check_c/REPORTS/` to the repo.

In [ ]:
drive.flush_and_unmount()
print('Drive flushed')